# UPDATED — ECOM national PSKU backtest with merge checks

**Code version: MERGE_CHECKED_V2**

Every lookup merge checks its row count. Depot expansion checks the exact expected number of rows. National measures are removed from the depot output to prevent inflated totals. Run all cells from the beginning to replace the functions and DataFrames already in the kernel.


One independent conversion for each month-end run from **2026-03-31 through 2026-07-31**. Forecasts are national PSKU forecasts; chains are aggregated before conversion. M is the run month, matching the original notebook convention.

The primary balance is unchanged: `max(forecast + next-month safety stock - opening stock, 0)`. Closing stock carries forward within each run/PSKU. Norm days remain clipped to 5–30 days. Depot output uses frozen prior-three-month primary contributions.

All predictive history and stock are restricted to dates **before the first day of the run month**. Future primary actuals are attached only for comparison. Forecast index rates are used for values; current master data/realignment and subsequently revised actuals are not historical snapshots. The original live notebook is preserved alongside this file as `*_live_backup.ipynb`.

In [ ]:
CODE_VERSION = 'MERGE_CHECKED_V2'
print(f'Running notebook version: {CODE_VERSION}')

from pathlib import Path
import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthEnd

BASE_DIR = Path('/data/aman_singh/acuuracy_check')
RUN_MONTHS = pd.date_range('2026-03-31', '2026-07-31', freq=MonthEnd())
FORECAST_PATH = BASE_DIR / 'Heuristics_all_combination_ecom_backtest.xlsx'
FORECAST_SHEET = 'Base'
FORECAST_VALUE_COLUMN = 'final Heuristic Value'  # crore; same selected heuristic as live notebook
FORECAST_RATE_COLUMN = 'qtr_ind_rate_x'
SOH_PATH = BASE_DIR / 'soh_base_oct_run.csv'  # raw material-level stock history through September closing
HISTORY_START = pd.Timestamp('2024-12-01')  # original norm window start
OUTPUT_DIR = BASE_DIR / 'FINAL ECOM National PSKU OTP' / 'Mar26_Jul26 Backtest'
EXCLUDED_PSKUS = {715096}
EXCLUDED_BRANDS = {'NC FREE', 'HC FREE'}

merge_audit = []


def checked_left_merge(left, right, on, label, **kwargs):
    keys = [on] if isinstance(on, str) else on
    if right.duplicated(keys).any():
        sample = right.loc[right.duplicated(keys, keep=False), keys].head(10)
        raise ValueError(f'{label}: duplicate right-side join keys would multiply rows:\n{sample}')
    merged = left.merge(right, on=on, how='left', validate='many_to_one', **kwargs)
    if len(merged) != len(left):
        raise AssertionError(f'{label}: row count changed from {len(left)} to {len(merged)}')
    merge_audit.append({'Merge': label, 'Left Rows': len(left), 'Right Rows': len(right),
                        'Expected Rows': len(left), 'Output Rows': len(merged), 'Expansion Allowed': False})
    return merged


## Load and validate the national forecast input

The supplied workbook has 240 zero-value/zero-index-rate rows across six PSKUs. These cannot be converted to reporting volume. They are excluded explicitly and saved in `Excluded Forecast Rows.csv`; other invalid values stop execution. Correct their rates upstream to include them.

Incomplete PSKU/run horizons are also excluded as entire groups and audited, because stock cannot be carried through absent forecast months without inventing demand. Complete horizons continue normally.

In [ ]:
def month_end(values):
    return pd.to_datetime(values, errors='raise') + MonthEnd(0)


def load_forecasts():
    frame = pd.read_excel(FORECAST_PATH, sheet_name=FORECAST_SHEET,
        usecols=lambda name: name in {'parent_material_code', 'brand_code', 'run_month', 'run_month_x',
                                    'month_date', FORECAST_VALUE_COLUMN, FORECAST_RATE_COLUMN})
    frame.columns = frame.columns.str.strip()
    if 'run_month' not in frame and 'run_month_x' in frame:
        frame = frame.rename(columns={'run_month_x': 'run_month'})
    needed = ['parent_material_code', 'brand_code', 'run_month', 'month_date',
              FORECAST_VALUE_COLUMN, FORECAST_RATE_COLUMN]
    missing = set(needed) - set(frame.columns)
    if missing:
        raise ValueError(f'Missing forecast columns: {sorted(missing)}')
    frame = frame.loc[frame['parent_material_code'].notna(), needed].copy()
    for col in ['run_month', 'month_date']:
        frame[col] = month_end(frame[col])
    frame = frame[frame['run_month'].isin(RUN_MONTHS)].copy()
    if set(RUN_MONTHS) - set(frame['run_month']):
        raise ValueError('The forecast file must contain all five requested run months')
    frame['parent_material_code'] = pd.to_numeric(frame['parent_material_code'], errors='raise').astype('int64')
    frame = frame[~frame['parent_material_code'].isin(EXCLUDED_PSKUS)
                  & ~frame['brand_code'].isin(EXCLUDED_BRANDS)].copy()
    frame = frame[frame['month_date'] >= frame['run_month']].copy()
    for col in [FORECAST_VALUE_COLUMN, FORECAST_RATE_COLUMN]:
        frame[col] = pd.to_numeric(frame[col], errors='raise')
    # Zero-valued rows with a zero rate cannot establish a reporting volume.
    # Keep an explicit audit export and exclude these rows rather than calculating 0/0.
    zero_rate = frame[FORECAST_RATE_COLUMN].eq(0) & frame[FORECAST_VALUE_COLUMN].eq(0)
    global excluded_forecasts
    excluded_forecasts = frame.loc[zero_rate].copy()
    excluded_forecasts['Exclusion Reason'] = 'Zero index rate: reporting volume cannot be established'
    if not excluded_forecasts.empty:
        print(f'Excluded {len(excluded_forecasts)} zero-rate/zero-value rows '
              f'({excluded_forecasts.parent_material_code.nunique()} PSKUs); see exclusion audit export')
    frame = frame.loc[~zero_rate].copy()
    if (frame[FORECAST_RATE_COLUMN].isna().any()
            or (frame[FORECAST_RATE_COLUMN] <= 0).any()
            or frame[FORECAST_VALUE_COLUMN].isna().any()
            or not np.isfinite(frame[[FORECAST_VALUE_COLUMN, FORECAST_RATE_COLUMN]]).all().all()):
        raise ValueError('Forecast values must be finite and rates must be positive; recalculate Excel formulas if needed')
    if (frame[FORECAST_VALUE_COLUMN] < 0).any():
        raise ValueError('Negative forecasts require review before conversion')
    grain = ['parent_material_code', 'run_month', 'month_date']
    if frame.duplicated(grain).any():
        raise ValueError('Expected one national forecast per PSKU/run/target month; chain rows must not be broadcast')
    frame['Offtake PSKU Forecast Vol'] = frame[FORECAST_VALUE_COLUMN] * 10**7 / frame[FORECAST_RATE_COLUMN]
    frame = frame.rename(columns={FORECAST_RATE_COLUMN: 'Index Rate'})
    incomplete = []
    for (run, psku), group in frame.groupby(['run_month', 'parent_material_code']):
        expected = pd.date_range(run, group['month_date'].max(), freq=MonthEnd())
        if set(group['month_date']) != set(expected):
            incomplete.append((run, psku))
        if group['brand_code'].nunique(dropna=False) != 1:
            raise ValueError(f'Inconsistent brand for PSKU {psku}')
    if incomplete:
        bad_groups = pd.MultiIndex.from_tuples(incomplete, names=['run_month', 'parent_material_code'])
        bad_mask = pd.MultiIndex.from_frame(frame[['run_month', 'parent_material_code']]).isin(bad_groups)
        rejected = frame.loc[bad_mask].copy()
        rejected['Exclusion Reason'] = 'Incomplete forecast horizon: cannot carry stock through missing months'
        excluded_forecasts = pd.concat([excluded_forecasts, rejected], ignore_index=True)
        frame = frame.loc[~bad_mask].copy()
        print(f'Excluded {len(incomplete)} incomplete PSKU/run horizons; see exclusion audit export')
    if set(RUN_MONTHS) - set(frame['run_month']):
        raise ValueError('No convertible forecasts remain for one or more requested run months')
    return frame.sort_values(grain).reset_index(drop=True)

forecasts = load_forecasts()
forecasts.groupby('run_month').agg(rows=('parent_material_code', 'size'),
                                 pskus=('parent_material_code', 'nunique'),
                                 first_target=('month_date', 'min'), last_target=('month_date', 'max'))

## Connections and historical sources
Database reads remain read-only. Offtake history is fetched separately from each run’s saved chain PSKU snapshot, then aggregated nationally. Stock dates are preserved; future stock is never relabelled as historical stock.

In [ ]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')

In [ ]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['ECOM', 'ECOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [ ]:
plan_actuals_query = """
SELECT
    CASE 
        WHEN MCM.chain = 'Big basket B2C' THEN 'Big Basket'
        WHEN MCM.chain = 'Flipkart-National' THEN 'Flipkart National'
        WHEN MCM.chain = 'Flipkart-Minutes' THEN 'Flipkart National'
        WHEN MCM.chain = 'Nykaa' THEN 'Nykaa'
        WHEN MCM.chain = 'Myntra' THEN 'Myntra'
        WHEN MCM.chain = 'MYNTRA' THEN 'Myntra'
        WHEN MCM.chain = 'Amazon B2C' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'ARIPL' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'RK WORLDINFOCOM' THEN 'Amazon RK'
        WHEN MCM.chain = 'RKWorld' THEN 'Amazon RK'
        WHEN MCM.chain = 'Flipkart-Grocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'FlipkartGrocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'Firstcry' THEN 'First Cry'
        WHEN MCM.chain = 'CITIMALL' THEN 'City Mall'
        WHEN MCM.chain = 'Meesho' THEN 'Meesho'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    MESR.month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
WHERE
    MESR.month_date > '2022-12-31'
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""
# Detect join multiplication in SQL before aggregation can hide it.
_checked_sql_dimensions = set()


def assert_sql_join_keys_unique(query, aliases):
    import re
    dimensions = {match.group(2).upper(): match.group(1) for match in re.finditer(
        r'JOIN\s*\(\s*(.*?)\)\s*(MCM|MM|CM)\s+ON', query,
        flags=re.IGNORECASE | re.DOTALL)}
    for alias, key in aliases:
        if alias not in dimensions:
            raise ValueError(f'Cannot locate SQL dimension {alias} for join validation')
        dimension = dimensions[alias]
        signature = (dimension.strip(), key)
        if signature in _checked_sql_dimensions:
            continue
        duplicate_query = f"""SELECT {key}, COUNT(*) AS join_rows
FROM ({dimension}) dimension_check
GROUP BY {key}
HAVING COUNT(*) > 1"""
        duplicates = pd.read_sql(duplicate_query, prod_conn)
        if not duplicates.empty:
            raise ValueError(f'SQL join {alias}: duplicate {key} would multiply sales rows:\n{duplicates.head(10)}')
        _checked_sql_dimensions.add(signature)


assert_sql_join_keys_unique(plan_actuals_query, [('MCM', 'customer'), ('MM', 'material_code')])

plan_actuals_df = pd.read_sql(plan_actuals_query, prod_conn)
plan_actuals_df.columns = plan_actuals_df.columns.str.lower()
plan_actuals_df['month_date'] = month_end(plan_actuals_df['month_date'])
plan_actuals_df = realign_pskus(plan_actuals_df.copy(), 'parent_material_code')
volume_cols = ['pri_actuals_vol_rum', 'sec_actuals_vol_rum']
plan_actuals_df[volume_cols] = plan_actuals_df[volume_cols].clip(lower=0)
primary_actuals = plan_actuals_df.groupby(['parent_material_code', 'month_date'], as_index=False)[volume_cols].sum()

In [ ]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""
assert_sql_join_keys_unique(depot_psku_primary_query,
    [('MCM', 'customer'), ('MM', 'material_code'), ('CM', 'customer_code')])
depot_actuals = pd.read_sql(depot_psku_primary_query, prod_conn)
depot_actuals.columns = depot_actuals.columns.str.lower()
depot_actuals['month_date'] = month_end(depot_actuals['month_date'])
depot_actuals = realign_pskus(depot_actuals.copy(), 'parent_material_code')
depot_actuals['pri_actuals_vol_rum'] = depot_actuals['pri_actuals_vol_rum'].clip(lower=0)
depot_actuals = depot_actuals.groupby(['depot_code', 'parent_material_code', 'month_date'], as_index=False)['pri_actuals_vol_rum'].sum()

In [ ]:
def prepare_stock(raw_stock, material_master):
    stock = raw_stock.copy()
    stock.columns = stock.columns.str.strip().str.lower()
    master = material_master.copy()
    master.columns = master.columns.str.lower()
    required = {'chain', 'material_code', 'date', 'soh'}
    if required - set(stock):
        raise ValueError(f'Stock file must contain {sorted(required)}')
    stock['chain'] = stock['chain'].replace({'MYNTRA': 'Myntra', 'BB': 'Big Basket',
        'Flipkart Mintues': 'Flipkart National', 'Flipkart Minutes': 'Flipkart National',
        'Flipkart-Grocery': 'Flipkart Grocery'})
    # The raw file also contains quick-commerce stock; keep only ECOM chains.
    ecom_chains = {'Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
        'Flipkart National', 'Myntra', 'Nykaa', 'Meesho', 'Purplle', 'Dealshare',
        'City Mall', '1MG', 'First Cry', 'Firstcry', 'FATEHPURIA HYGIENE', 'EMAZING DEALS'}
    stock = stock[stock['chain'].isin(ecom_chains)].copy()
    stock = stock[stock['material_code'].astype(str).str.strip().ne('Combo')].copy()
    stock['material_code'] = pd.to_numeric(stock['material_code'], errors='raise').astype('int64')
    master['material_code'] = pd.to_numeric(master['material_code'], errors='raise').astype('int64')
    stock_rows_before_merge = len(stock)
    stock = checked_left_merge(stock[['chain', 'material_code', 'date', 'soh']],
        master[['material_code', 'parent_material_code', 'vol_per_unit', 'uom_reporting']],
        on='material_code', label='Stock to material master', indicator=True)
    assert len(stock) == stock_rows_before_merge, 'Material merge multiplied stock rows'
    unmatched = stock['_merge'].ne('both') | stock['parent_material_code'].isna()
    if unmatched.any():
        raise ValueError(f'{unmatched.sum()} ECOM stock rows lack material-master mappings')
    stock['snapshot_date'] = pd.to_datetime(stock['date'], errors='raise').dt.normalize()
    # An opening snapshot on the 1st represents the preceding month's closing stock.
    # Preserve its actual timestamp for audit; never cap later dates to a run cutoff.
    stock['as_on_date'] = stock['snapshot_date'] - pd.to_timedelta(stock['snapshot_date'].dt.day.eq(1).astype(int), unit='D')
    stock['soh'] = pd.to_numeric(stock['soh'], errors='raise')
    stock['vol_per_unit'] = pd.to_numeric(stock['vol_per_unit'], errors='raise')
    if stock[['soh', 'vol_per_unit', 'uom_reporting']].isna().any().any():
        raise ValueError('Stock quantities or material volume factors are missing')
    stock['current_soh'] = stock['soh'] * stock['vol_per_unit'] / np.where(
        stock['uom_reporting'].isin(['KL', 'TO']), 10**6, 10**3)
    if not np.isfinite(stock['current_soh']).all():
        raise ValueError('Nonfinite stock volumes')
    stock = realign_pskus(stock.copy(), 'parent_material_code')
    stock = stock[~stock['parent_material_code'].isin(EXCLUDED_PSKUS)].copy()
    stock = stock.groupby(['chain', 'parent_material_code', 'as_on_date', 'snapshot_date'], as_index=False)['current_soh'].sum()
    stock['stock_month'] = month_end(stock['as_on_date'])
    nonzero = stock.groupby(['chain', 'as_on_date', 'snapshot_date'])['current_soh'].transform('sum').ne(0)
    return stock[nonzero].copy()


material_master_df = pd.read_sql("""SELECT material_code, parent_material_code,
    vol_per_unit, uom_reporting FROM mst_material
    WHERE company_code='MIL' AND latest_record_ind=1""", prod_conn)
soh_df = prepare_stock(pd.read_csv(SOH_PATH, low_memory=False), material_master_df)


def load_run_offtakes(run_month):
    query = f"""SELECT * FROM TRN_DF_ECOM_OFFTAKE_PSKU
        WHERE run_month = '{run_month:%Y-%m-%d}'
          AND month_date >= '{HISTORY_START:%Y-%m-%d}'
          AND month_date < '{run_month.replace(day=1):%Y-%m-%d}'"""
    history = pd.read_sql(query, dev_conn)
    history.columns = history.columns.str.lower()
    if history.empty:
        raise ValueError(f'No saved offtake history for run {run_month.date()}')
    history['month_date'] = month_end(history['month_date'])
    history = realign_pskus(history.copy(), 'parent_material_code')
    history['vol_in_rum'] = pd.to_numeric(history['vol_in_rum'], errors='raise')
    if not np.isfinite(history['vol_in_rum']).all() or (history['vol_in_rum'] < 0).any():
        raise ValueError(f'Invalid historical offtake for run {run_month.date()}')
    assert (history['month_date'] < run_month.replace(day=1)).all()
    return history.groupby(['parent_material_code', 'month_date'], as_index=False)['vol_in_rum'].sum()

## Conversion and depot allocation

Stock comes from `soh_base_oct_run.csv`. Raw material quantities are converted to reporting volume using the same material-master formula as the live notebook; quick-commerce chains are excluded. Snapshots dated the first of a month represent the previous month's closing stock (e.g. July 1 → June 30), while their original dates remain in `snapshot_date`. Later snapshots are never capped or backdated to a run cutoff.

For each run, norm inventory uses the mean national monthly closing stock since December 2024. Each chain contributes its latest snapshot in that month. Opening stock uses each chain's last snapshot in the immediately preceding month. Missing stock is explicitly flagged and defaults to zero, as in the live balance; missing norms default to five days. Older months are not substituted for a missing opening-stock month.

The last supplied forecast month has no following-month forecast, so its closing-stock target is zero, preserving the live notebook's boundary convention; `Safety Stock Horizon Missing` identifies it. Depot shares use the preceding three calendar months only. PSKUs without positive P3M sales use equal shares among their historically known depots; PSKUs without a historical depot remain `UNALLOCATED`. National forecasts already cover the supplied PSKU scope, so the old additive “other chains” forecast is removed to prevent double counting.

For the July 2026 run, opening stock is selected only from June closing stock, including the July 1 opening snapshot where present. Norm estimation uses historical stock only through June.

In [ ]:
def run_conversion(run, run_forecasts, history, stock, actuals, depots):
    cutoff = run.replace(day=1)
    prior_month = run - MonthEnd(1)
    history = history[(history['month_date'] >= HISTORY_START) & (history['month_date'] < cutoff)].copy()
    stock = stock[(stock['as_on_date'] >= HISTORY_START) & (stock['as_on_date'] < cutoff)].copy()
    if history.empty or stock.empty:
        raise ValueError(f'Missing pre-run history or inventory for {run.date()}')
    # Latest chain snapshot in each month; sum PSKUs, then chains to national stock.
    snapshot_col = 'snapshot_date' if 'snapshot_date' in stock else 'as_on_date'
    latest = stock.groupby(['chain', 'stock_month'])[snapshot_col].transform('max')
    closing = stock[stock[snapshot_col].eq(latest)].copy()
    monthly_stock = closing.groupby(['parent_material_code', 'stock_month'], as_index=False)['current_soh'].sum()
    average_stock = monthly_stock.groupby('parent_material_code')['current_soh'].mean()
    opening = monthly_stock[monthly_stock['stock_month'].eq(prior_month)].set_index('parent_material_code')['current_soh']
    # Same calendar denominator for every PSKU, including zero offtake months.
    days = (cutoff - HISTORY_START).days
    daily_offtake = history.groupby('parent_material_code')['vol_in_rum'].sum() / days
    norm_days = (average_stock / daily_offtake.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).clip(5, 30)
    outputs = []
    for psku, group in run_forecasts.groupby('parent_material_code'):
        group = group.sort_values('month_date').copy()
        if group.iloc[0]['month_date'] != run:
            raise ValueError(f'Missing M forecast for {run.date()}, PSKU {psku}')
        nd = norm_days.get(psku, np.nan)
        missing_norm = pd.isna(nd)
        nd = 5.0 if missing_norm else float(nd)
        group['Norm Days'] = nd
        group['Norms SOH'] = group['Offtake PSKU Forecast Vol'] * nd / group['month_date'].dt.days_in_month
        group['Safety Stock Vol'] = group['Norms SOH'].shift(-1).fillna(0)
        group['Safety Stock Horizon Missing'] = group['month_date'].eq(group['month_date'].max())
        missing_opening = psku not in opening.index
        prev_soh = max(float(opening.get(psku, 0)), 0)
        for _, row in group.iterrows():
            forecast = row['Offtake PSKU Forecast Vol']
            primary = max(forecast + row['Safety Stock Vol'] - prev_soh, 0)
            closing_soh = max(prev_soh + primary - forecast, 0)
            record = row.to_dict()
            record.update({'Opening SOH Vol': prev_soh, 'Calculated Primary Vol': primary,
                'Final Assumed Closing SOH Vol': closing_soh, 'Opening SOH Missing': missing_opening,
                'Norm History Missing': missing_norm, 'History Cutoff': prior_month})
            outputs.append(record)
            prev_soh = closing_soh
    result = pd.DataFrame(outputs)
    # Evaluation actuals never enter the forecast balance.
    national_rows_before_merge = len(result)
    result = checked_left_merge(result, actuals, on=['parent_material_code', 'month_date'],
                                label=f'{run.date()}: national actuals')
    assert len(result) == national_rows_before_merge, 'Actuals merge multiplied national rows'
    result = result.rename(columns={'pri_actuals_vol_rum': 'Primary Actuals Vol', 'sec_actuals_vol_rum': 'Sec Actuals Vol'})
    result['M month'] = 'M+' + ((result['month_date'].dt.year-run.year)*12 + result['month_date'].dt.month-run.month).astype(str)
    result.loc[result['month_date'].eq(run), 'M month'] = 'M'
    for col in ['Offtake PSKU Forecast Vol', 'Calculated Primary Vol', 'Primary Actuals Vol', 'Sec Actuals Vol']:
        result[col[:-3] + 'Val'] = result[col] * result['Index Rate'] / 10**7
    result['Error Vol'] = result['Calculated Primary Vol'] - result['Primary Actuals Vol']
    result['Abs Error Vol'] = result['Error Vol'].abs()
    # Freeze contributions at the run cutoff; use an exact three-month window.
    past = depots[depots['month_date'] < cutoff].copy()
    recent = past[past['month_date'] >= cutoff - pd.DateOffset(months=3)]
    weights = recent.groupby(['parent_material_code', 'depot_code'], as_index=False)['pri_actuals_vol_rum'].sum()
    totals = weights.groupby('parent_material_code')['pri_actuals_vol_rum'].transform('sum')
    weights['PSKU P3M Contribution'] = weights['pri_actuals_vol_rum'] / totals.replace(0, np.nan)
    weights = weights[weights['PSKU P3M Contribution'].notna()].copy()
    weights['Allocation Method'] = 'P3M primary'
    missing = set(result['parent_material_code']) - set(weights['parent_material_code'])
    fallback = past[past['parent_material_code'].isin(missing)][['parent_material_code', 'depot_code']].drop_duplicates()
    fallback['PSKU P3M Contribution'] = 1 / fallback.groupby('parent_material_code')['depot_code'].transform('size')
    fallback['Allocation Method'] = 'Equal historical depots'
    unallocated = missing - set(fallback['parent_material_code'])
    unallocated = pd.DataFrame([{'parent_material_code': p, 'depot_code': 'UNALLOCATED',
        'PSKU P3M Contribution': 1.0, 'Allocation Method': 'Unallocated'} for p in unallocated])
    weights = pd.concat([weights, fallback, unallocated], ignore_index=True)
    weight_keys = ['parent_material_code', 'depot_code']
    if weights.duplicated(weight_keys).any():
        raise ValueError(f'{run.date()}: duplicate PSKU/depot allocation weights')
    share_totals = weights.groupby('parent_material_code')['PSKU P3M Contribution'].sum()
    if not np.allclose(share_totals, 1):
        raise ValueError(f'{run.date()}: depot shares must sum to one per PSKU')
    depot_counts = weights.groupby('parent_material_code').size()
    expected_counts = result['parent_material_code'].map(depot_counts)
    if expected_counts.isna().any():
        raise ValueError(f'{run.date()}: forecast PSKU has no depot allocation')
    expected_rows = int(expected_counts.sum())
    allocated = result.merge(weights[['parent_material_code', 'depot_code', 'PSKU P3M Contribution', 'Allocation Method']],
                             on='parent_material_code', how='left', validate='many_to_many')
    if len(allocated) != expected_rows:
        raise AssertionError(f'{run.date()}: unexpected row growth in depot allocation')
    if allocated.duplicated(['parent_material_code', 'depot_code', 'run_month', 'month_date']).any():
        raise ValueError(f'{run.date()}: duplicate forecast rows after depot allocation')
    merge_audit.append({'Merge': f'{run.date()}: expand PSKUs to depots', 'Left Rows': len(result),
        'Right Rows': len(weights), 'Expected Rows': expected_rows, 'Output Rows': len(allocated),
        'Expansion Allowed': True})
    allocated['Calculated Depot PSKU Primary Vol'] = allocated['Calculated Primary Vol'] * allocated['PSKU P3M Contribution']
    allocated['Calculated Depot PSKU Primary Val'] = allocated['Calculated Depot PSKU Primary Vol'] * allocated['Index Rate'] / 10**7
    reconciliation = allocated.groupby(['parent_material_code', 'run_month', 'month_date'])['Calculated Depot PSKU Primary Vol'].sum()
    national = result.set_index(['parent_material_code', 'run_month', 'month_date'])['Calculated Primary Vol']
    assert np.allclose(reconciliation.reindex(national.index), national)
    assert not result.duplicated(['parent_material_code', 'run_month', 'month_date']).any()
    assert np.isfinite(result['Calculated Primary Vol']).all()
    assert (result['Calculated Primary Vol'] >= 0).all()
    # A depot row must not carry summable copies of national measures.
    allocated['Allocated Depot PSKU Offtake Forecast Vol'] = allocated['Offtake PSKU Forecast Vol'] * allocated['PSKU P3M Contribution']
    allocated['Allocated Depot PSKU Offtake Forecast Val'] = allocated['Allocated Depot PSKU Offtake Forecast Vol'] * allocated['Index Rate'] / 10**7
    allocated = allocated[[
        'parent_material_code', 'brand_code', 'run_month', 'month_date', 'depot_code',
        'Index Rate', 'M month', 'History Cutoff', 'PSKU P3M Contribution', 'Allocation Method',
        'Calculated Depot PSKU Primary Vol', 'Calculated Depot PSKU Primary Val',
        'Allocated Depot PSKU Offtake Forecast Vol', 'Allocated Depot PSKU Offtake Forecast Val'
    ]].copy()
    # Join actual depot sales for evaluation; never repeat national actuals at depots.
    depot_evaluation = depots[['parent_material_code', 'depot_code', 'month_date', 'pri_actuals_vol_rum']].copy()
    depot_rows_before_actuals_merge = len(allocated)
    allocated = checked_left_merge(allocated, depot_evaluation,
        on=['parent_material_code', 'depot_code', 'month_date'], label=f'{run.date()}: depot actuals')
    assert len(allocated) == depot_rows_before_actuals_merge, 'Actuals merge multiplied depot rows'
    allocated = allocated.rename(columns={'pri_actuals_vol_rum': 'Depot PSKU Primary Actuals Vol'})
    allocated['Depot PSKU Primary Actuals Val'] = allocated['Depot PSKU Primary Actuals Vol'] * allocated['Index Rate'] / 10**7
    for national_col, depot_col in [
        ('Calculated Primary Val', 'Calculated Depot PSKU Primary Val'),
        ('Offtake PSKU Forecast Val', 'Allocated Depot PSKU Offtake Forecast Val')
    ]:
        expected = result.set_index(['parent_material_code', 'run_month', 'month_date'])[national_col]
        observed = allocated.groupby(['parent_material_code', 'run_month', 'month_date'])[depot_col].sum()
        assert np.allclose(observed.reindex(expected.index), expected)
    forbidden_national_columns = {'Calculated Primary Val', 'Calculated Primary Vol',
        'final Heuristic Value', 'Primary Actuals Val', 'Primary Actuals Vol'}
    assert not forbidden_national_columns.intersection(allocated.columns), 'National measures leaked into depot output'
    return result, allocated

## Run all five backtests

In [ ]:
# Keep the initial stock merge audit, replace prior run audits on a cell rerun.
merge_audit[:] = [row for row in merge_audit if row['Merge'] == 'Stock to material master']
national_runs, depot_runs = [], []
for run in RUN_MONTHS:
    history = load_run_offtakes(run)
    national, allocated = run_conversion(run, forecasts[forecasts['run_month'].eq(run)].copy(),
                                        history, soh_df, primary_actuals, depot_actuals)
    national_runs.append(national)
    depot_runs.append(allocated)

rename = {'parent_material_code': 'PSKU', 'brand_code': 'Brand', 'run_month': 'Run Month',
          'month_date': 'Month Date', 'depot_code': 'Depot'}
final_df = pd.concat(national_runs, ignore_index=True).rename(columns=rename)
final_depot_psku_df = pd.concat(depot_runs, ignore_index=True).rename(columns=rename)
assert set(final_df['Run Month']) == set(RUN_MONTHS)
assert not final_df.duplicated(['PSKU', 'Run Month', 'Month Date']).any()
assert not final_depot_psku_df.duplicated(['PSKU', 'Depot', 'Run Month', 'Month Date']).any()
merge_audit_df = pd.DataFrame(merge_audit)
assert merge_audit_df['Expected Rows'].eq(merge_audit_df['Output Rows']).all()
print(f'{CODE_VERSION}: all merge row-count checks passed')
display(merge_audit_df)
summary = final_df.groupby('Run Month').agg(
    rows=('PSKU', 'size'), pskus=('PSKU', 'nunique'),
    primary_volume=('Calculated Primary Vol', 'sum'),
    missing_opening_stock=('Opening SOH Missing', 'sum'),
    missing_norm_history=('Norm History Missing', 'sum'))
summary

## Compare national totals and depot allocations

Use `final_df` for national PSKU totals. Depot output contains only allocated forecasts/primary and actual depot sales, with explicit depot column names. National values are not repeated in depot rows.

The stock balance can raise primary above offtake in M: offtake + required closing stock − opening stock, floored at zero for each PSKU. The table below exposes those components and missing-opening-stock counts.

Every lookup merge enforces unique right-side keys and unchanged row counts. PSKU-to-depot expansion is intentional and checked against the exact number of depots per PSKU. `Merge Row Count Audit.csv` records expected and observed rows. SQL dimension keys are checked before sales aggregation, so duplicate dimension matches cannot silently inflate actuals.

In [ ]:
national_check = final_df.copy()
for col in ['Opening SOH Vol', 'Safety Stock Vol', 'Final Assumed Closing SOH Vol']:
    national_check[col[:-3] + 'Val'] = national_check[col] * national_check['Index Rate'] / 10**7
national_comparison = national_check.groupby(['Run Month', 'Month Date'])[
    ['final Heuristic Value', 'Calculated Primary Val', 'Primary Actuals Val',
     'Opening SOH Val', 'Safety Stock Val', 'Final Assumed Closing SOH Val']
].sum(min_count=1)
depot_comparison = final_depot_psku_df.groupby(['Run Month', 'Month Date'])[
    ['Calculated Depot PSKU Primary Val', 'Allocated Depot PSKU Offtake Forecast Val']
].sum(min_count=1)
assert np.allclose(national_comparison['Calculated Primary Val'],
                   depot_comparison['Calculated Depot PSKU Primary Val'].reindex(national_comparison.index))
assert np.allclose(national_comparison['final Heuristic Value'],
                   depot_comparison['Allocated Depot PSKU Offtake Forecast Val'].reindex(national_comparison.index))
national_comparison['Missing Opening Stock PSKUs'] = national_check.groupby(
    ['Run Month', 'Month Date'])['Opening SOH Missing'].sum()
national_comparison

## Save national PSKU and depot PSKU outputs
Missing evaluation actuals are preserved as missing, rather than treated as zero. Saved forecasts contain run and target months for backtest joins. No chain-level forecast is fabricated from national input.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUTPUT_DIR / 'ECOM National PSKU Primary_Mar26_Jul26 Backtest.csv', index=False)
final_depot_psku_df.to_csv(OUTPUT_DIR / 'ECOM Depot PSKU Primary_Mar26_Jul26 Backtest.csv', index=False)
excluded_forecasts.to_csv(OUTPUT_DIR / 'Excluded Forecast Rows.csv', index=False)
soh_df.to_csv(OUTPUT_DIR / 'Prepared SOH History.csv', index=False)
merge_audit_df.to_csv(OUTPUT_DIR / 'Merge Row Count Audit.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'Backtest Run Summary.csv')
print(f'Saved backtest outputs to {OUTPUT_DIR}')